In [ ]:
import argparse
import datetime
from pyexpat import model
import numpy as np
import time
import torch.backends.cudnn as cudnn
import json
import os
import pandas as pd
from pathlib import Path
import torch.distributed as dist
from sklearn.tests.test_multiclass import n_classes
from tqdm import tqdm
from collections import OrderedDict
from timm.data.mixup import Mixup
from timm.models import create_model
from timm.loss import LabelSmoothingCrossEntropy, SoftTargetCrossEntropy
from utils import FocalLoss, DynamicFocalLoss,GHMC,MarginLoss, BinaryFocalLoss
from timm.utils import ModelEma
from utils import create_optimizer, get_parameter_groups, LayerDecayValueAssigner
from utils import NativeScalerWithGradNormCount as NativeScaler
import utils
from scipy import interpolate
import task_model
import math
import sys
from typing import Iterable, Optional
import torch
import torch.nn.functional as F
from timm.utils import ModelEma
import utils
from einops import rearrange

import warnings
warnings.filterwarnings("ignore")

In [ ]:
def get_args():
    parser = argparse.ArgumentParser('Fine-tuning and evaluation script for EEG classification', add_help=False)
    parser.add_argument('--batch_size', default=64, type=int)
    parser.add_argument('--epochs', default=30, type=int)
    parser.add_argument('--update_freq', default=1, type=int)
    parser.add_argument('--save_ckpt_freq', default=1, type=int)
    parser.add_argument('--distributed', default=True, type= bool, help='Enabling distributed')

    # robust evaluation
    parser.add_argument('--robust_test', default=None, type=str,
                        help='robust evaluation dataset')

    # Model parameters
    parser.add_argument('--model', default='base_patch200_200', type=str, metavar='MODEL',
                        help='Name of model to train')
    parser.add_argument('--qkv_bias', action='store_true')
    parser.add_argument('--disable_qkv_bias', action='store_false', dest='qkv_bias')
    parser.set_defaults(qkv_bias=True)
    parser.add_argument('--rel_pos_bias', action='store_true')
    parser.add_argument('--disable_rel_pos_bias', action='store_false', dest='rel_pos_bias')
    parser.set_defaults(rel_pos_bias=True)
    parser.add_argument('--abs_pos_emb', action='store_true')
    parser.set_defaults(abs_pos_emb=False)
    parser.add_argument('--layer_scale_init_value', default=0.1, type=float,
                        help="0.1 for base, 1e-5 for large. set 0 to disable layer scale")

    parser.add_argument('--input_size', default=200, type=int,
                        help='EEG input size')

    parser.add_argument('--drop', type=float, default=0.0, metavar='PCT',
                        help='Dropout rate (default: 0.)')
    parser.add_argument('--attn_drop_rate', type=float, default=0.0, metavar='PCT',
                        help='Attention dropout rate (default: 0.)')
    parser.add_argument('--drop_path', type=float, default=0.1, metavar='PCT',
                        help='Drop path rate (default: 0.1)')

    parser.add_argument('--disable_eval_during_finetuning', action='store_true', default=False)

    parser.add_argument('--model_ema', action='store_true', default=False)
    parser.add_argument('--model_ema_decay', type=float, default=0.9999, help='')
    parser.add_argument('--model_ema_force_cpu', action='store_true', default=False, help='')

    # Optimizer parameters
    parser.add_argument('--opt', default='adamw', type=str, metavar='OPTIMIZER',
                        help='Optimizer (default: "adamw"')
    parser.add_argument('--opt_eps', default=1e-8, type=float, metavar='EPSILON',
                        help='Optimizer Epsilon (default: 1e-8)')
    parser.add_argument('--opt_betas', default=None, type=float, nargs='+', metavar='BETA',
                        help='Optimizer Betas (default: None, use opt default)')
    parser.add_argument('--clip_grad', type=float, default=None, metavar='NORM',
                        help='Clip gradient norm (default: None, no clipping)')
    parser.add_argument('--momentum', type=float, default=0.9, metavar='M',
                        help='SGD momentum (default: 0.9)')
    parser.add_argument('--weight_decay', type=float, default=0.05,
                        help='weight decay (default: 0.05)')
    parser.add_argument('--weight_decay_end', type=float, default=None, help="""Final value of the
        weight decay. We use a cosine schedule for WD and using a larger decay by
        the end of training improves performance for ViTs.""")

    parser.add_argument('--lr', type=float, default=5e-4, metavar='LR',
                        help='learning rate (default: 5e-4)')
    parser.add_argument('--layer_decay', type=float, default=0.9)
    parser.add_argument('--warmup_lr', type=float, default=1e-6, metavar='LR',
                        help='warmup learning rate (default: 1e-6)')
    parser.add_argument('--min_lr', type=float, default=1e-6, metavar='LR',
                        help='lower lr bound for cyclic schedulers that hit 0 (1e-5)')

    parser.add_argument('--warmup_epochs', type=int, default=5, metavar='N',
                        help='epochs to warmup LR, if scheduler supports')
    parser.add_argument('--warmup_steps', type=int, default=-1, metavar='N',
                        help='num of steps to warmup LR, will overload warmup_epochs if set > 0')

    parser.add_argument('--smoothing', type=float, default=0.1,
                        help='Label smoothing (default: 0.1)')
    parser.add_argument('--focalloss', action='store_true', default=False)
    parser.add_argument('--focal_alpha', type=str, default='1.0 1.0 1.0', help='Focal Loss alpha')
    parser.add_argument('--focal_gamma', type=float, default=2, help='Focal Loss gamma')

    parser.add_argument('--marginloss', action='store_true', default=False)
    parser.add_argument('--margin', type=float, default=1, help='margin in margin loss')

    # * Random Erase params
    parser.add_argument('--reprob', type=float, default=0.25, metavar='PCT',
                        help='Random erase prob (default: 0.25)')
    parser.add_argument('--remode', type=str, default='pixel',
                        help='Random erase mode (default: "pixel")')
    parser.add_argument('--recount', type=int, default=1,
                        help='Random erase count (default: 1)')
    parser.add_argument('--resplit', action='store_true', default=False,
                        help='Do not random erase first (clean) augmentation split')

    # * Finetuning params
    parser.add_argument('--finetune', default='',
                        help='finetune from checkpoint')
    parser.add_argument('--model_key', default='model|module', type=str)
    parser.add_argument('--model_prefix', default='', type=str)
    parser.add_argument('--model_filter_name', default='gzp', type=str)
    parser.add_argument('--init_scale', default=0.001, type=float)
    parser.add_argument('--use_mean_pooling', action='store_true')
    parser.set_defaults(use_mean_pooling=True)
    parser.add_argument('--use_cls', action='store_false', dest='use_mean_pooling')
    parser.add_argument('--disable_weight_decay_on_rel_pos_bias', action='store_true', default=False)

    # Dataset parameters
    parser.add_argument('--nb_classes', default=0, type=int,
                        help='number of the classification types')

    parser.add_argument('--output_dir', default='',
                        help='path where to save, empty for no saving')
    parser.add_argument('--log_dir', default=None,
                        help='path where to tensorboard log')
    parser.add_argument('--device', default='cuda',
                        help='device to use for training / testing')
    parser.add_argument('--seed', default=0, type=int)
    parser.add_argument('--resume', default='',
                        help='resume from checkpoint')
    parser.add_argument('--auto_resume', action='store_true')
    parser.add_argument('--no_auto_resume', action='store_false', dest='auto_resume')
    parser.set_defaults(auto_resume=True)
    parser.add_argument('--task_model', default='',
                        help='resume the fine-tuned task model from checkpoint')

    parser.add_argument('--save_ckpt', action='store_true')
    parser.add_argument('--no_save_ckpt', action='store_false', dest='save_ckpt')
    parser.set_defaults(save_ckpt=True)

    parser.add_argument('--start_epoch', default=0, type=int, metavar='N',
                        help='start epoch')
    parser.add_argument('--eval', action='store_true',
                        help='Perform evaluation only')
    parser.add_argument('--dist_eval', action='store_true', default=False,
                        help='Enabling distributed evaluation')
    parser.add_argument('--num_workers', default=10, type=int)
    parser.add_argument('--pin_mem', action='store_true',
                        help='Pin CPU memory in DataLoader for more efficient (sometimes) transfer to GPU.')
    parser.add_argument('--no_pin_mem', action='store_false', dest='pin_mem')
    parser.set_defaults(pin_mem=True)

    parser.add_argument('--train_eeg_montage', default='average',
                        help='average (19) | bipolar (18) | combine (19 average + 18 bipolar)')
    parser.add_argument('--train_eeg_length', default=False, type=int,
                        help='15 | 10 | 5 Training samples length, use it for training IIIC (15s, 10s, 5s), but not for wave like spike. Training sample for IIIC is prepared with 15 seconds, and can be cut for assigned train_eeg_length; Spike samples should be prepared with fixed length like 1 seconds')
    parser.add_argument('--training_data_dir', default='',help='training data dir')

    parser.add_argument('--predict', action='store_true',
                        help='Perform evaluation only')
    parser.add_argument('--eval_list_file', default='', type=str,
                        help='a file to list test or evaluation dataset')
    parser.add_argument('--eval_list_column', default='',
                        help='file name column in test or evaluation dataset')
    parser.add_argument('--eval_results_dir', default='',
                        help='path where to save prediction results')
    parser.add_argument('--eeg_montage', default='average',
                        help='average (19) | bipolar (18) | combine (19 average + 18 bipolar) | single')

    parser.add_argument('--prediction_slipping_step', default=0, type=int,
                        help='slipping step in continuous prediction, if original hz>200 prediction_slipping_step better to be x/100 or xx/128')

    parser.add_argument('--prediction_slipping_step_second', default=0, type=int,
                        help='slipping step in second in continuous prediction')

    parser.add_argument('--data_format',  default='mat', type=str,
                        help='in continuous test, original data format mat | edf | pkl')

    parser.add_argument('--sampling_rate', default=0, type=int,
                        help='give sampling rate if mat/edf data files do not have this information')

    parser.add_argument('--already_format_channel_order', default='no',
                        choices=['yes','y', 'no','n'],type=str,
                        help='do not have channel info in data file, but the data if ordered by FP1, F3, C3, P3, F7, T3, T5, O1, FZ, CZ, PZ, FP2, F4, C4, P4, F8, T4, T6, O2')

    parser.add_argument('--already_average_montage', default='no',
                        choices=['yes','y', 'no','n'],type=str,
                        help='the original data already have average montage')

    parser.add_argument( '--allow_missing_channels', type=str,
                         choices=['yes','y', 'no','n'], default='no',
                         help='Allow missing channels (yes/no)')
    parser.add_argument('--polarity', default=1, type=int,
                        help='Set 1 ｜ -1, default 1. If set -1, the signal is inverted')

    parser.add_argument('--smooth_result',  default='', type=str,
                        help='smooth the continuous results by EMA')

    parser.add_argument('--need_spikes_10s_result', default='no', type=str,
                        choices=['yes', 'y', 'no', 'n'],
                        help='not only output 1-second prediction, but also output 10-second')

    parser.add_argument('--spikes_10s_result_slipping_step_second', default=0, type=int, help='slipping step (in second) in continuous results based on 10s spikes segments. Could set it as 10')

    parser.add_argument('--test_data_format',  default='pkl', type=str,
                        help='in test, original data format mat | pkl')

    parser.add_argument('--max_length_hour', default='no', type=str,
                        help='Only analyze the first n hours of data.')

    parser.add_argument('--rewrite_results', default='no', type=str,
                        choices=['yes', 'y', 'no', 'n'],
                        help='rewrite reults')

    parser.add_argument('--leave_one_hemisphere_out', default='no', type=str,
                        choices=['no', 'right', 'left', 'middle'],
                        help='set right or left or middle hemisphere 0')

    parser.add_argument('--channel_symmetric_flip', default='no', type=str,
                        choices=['no', 'right', 'left'],
                        help='symmetrically map the right(left) hemisphere data to the left(right) hemisphere.')


    # distributed training parameters
    parser.add_argument('--world_size', default=1, type=int,
                        help='number of distributed processes')
    parser.add_argument('--local_rank', default=-1, type=int)
    parser.add_argument('--dist_on_itp', action='store_true')
    parser.add_argument('--dist_url', default='env://',
                        help='url used to set up distributed training')
    parser.add_argument('--enable_deepspeed', action='store_true', default=False)

    parser.add_argument('--dataset', default='TUAB', type=str,
                        help='dataset: IIIC | SPIKES | NORMAL | SLOWING | BS  | SLEEP')
    parser.add_argument('--eval_sub_dir', type=str,
                        help='dataset: subject directory')

    known_args, _ = parser.parse_known_args()

    if known_args.enable_deepspeed:
        try:
            print("Use deepspeed==0.4.0'")
            import deepspeed
            from deepspeed import DeepSpeedConfig
            parser = deepspeed.add_config_arguments(parser)
            ds_init = deepspeed.initialize
        except:
            print("Please 'pip install deepspeed==0.4.0'")
            exit(0)
    else:
        ds_init = None

    return parser.parse_args(), ds_init